# Handling Skewed Data

Skewed data happens when certain keys or groups are disproportionately represented in the dataset, leading to unbalanced workloads.
Dealing with skewed data in spark is essential for ensuring that tasks are evenly distributed across partitions, thus avoiding performance bottlenecks.

In [ ]:
import time

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("Colab_PySpark_Test").getOrCreate()

## Salting

* What it is: When performing operations like joins or aggregations, if one key (e.g., a particular ID
or category) is highly skewed (one value appears much more frequently than others), you can
"salt" the key by adding a random suffix or prefix. This spreads the data more evenly across
partitions.
* How it works: For example, if you are joining on a "user_id" key, you could append a random number
to the key (e.g., user_id_1, user_id_2), which splits the skewed data across more
partitions. After the operation, you can remove the salt to return to the original data
structure.
* This can help avoid overloading a single partition with the majority of the data.

### Example 1:

#### Create Skewed Dataset

In [ ]:
from pyspark.sql.functions import lit, expr

customers = (
    spark.range(3000)
         .withColumn(
             "CustomerID",
             expr("""
                 CASE
                     WHEN id < 1000 THEN 'C1'
                     WHEN id < 2000 THEN 'C2'
                     ELSE 'C3'
                 END
             """)
         )
         .withColumn("CustomerName", lit("Customer"))
)

customers.count()

In [ ]:
from pyspark.sql.functions import expr

orders = (
    spark.range(10000000)
         .withColumn(
             "CustomerID",
             expr("""
                 CASE
                     WHEN id < 9000000 THEN 'C1'
                     WHEN id < 9500000 THEN 'C2'
                     ELSE 'C3'
                 END
             """)
         )
)

orders.count()

In [ ]:
spark.conf.get("spark.sql.shuffle.partitions")

In [ ]:
orders = orders.repartition(10)
customers = customers.repartition(10)

In [ ]:
from pyspark.sql.functions import spark_partition_id
orders.withColumn("partition_id", spark_partition_id()).groupBy("partition_Id").count().show(10)

In [ ]:
joined_df = (
    orders.hint("MERGE")
          .join(
              customers.hint("MERGE"),
              "CustomerID"
          )
)

In [ ]:
joined_df.explain("formatted")

In [ ]:
from pyspark.sql.functions import floor, rand

orders_salted = orders.withColumn(
    "salt",
    floor(rand() * 10)
)

In [ ]:
orders_salted.show(5)

In [ ]:
from pyspark.sql.functions import explode
from pyspark.sql.functions import array
from pyspark.sql.functions import lit

customers_salted = customers.withColumn(
    "salt",
    explode(
        array(*[lit(i) for i in range(10)])
    )
)

In [ ]:
customers_salted.show(5)

In [ ]:
salted_join = (
    orders_salted.join(
        customers_salted,
        ["CustomerID", "salt"]
    )
)

In [ ]:
salted_join.repartition(
    10,
    "CustomerID",
    "salt"
).withColumn(
    "partition_id",
    spark_partition_id()
).groupBy(
    "partition_id"
).count().orderBy(
    "partition_id"
).show(20, False)

#### Perform Join Without Salting

In [ ]:
start = time.time()
joined_df.count()
print(time.time()-start)

#### Perform Join With Salting

In [ ]:
start = time.time()
salted_join.count()
print(time.time()-start)

### Example 2:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat, lit, rand

#### Sample Skewed DataFrame

In [ ]:
sales_data = [
    ("India", 100),
    ("India", 200),
    ("India", 150),
    ("USA", 50),
    ("UK", 30)
]

sales_df = spark.createDataFrame(sales_data, ["country", "amount"])

country_data = [
    ("India", "Asia"),
    ("USA", "North America"),
    ("UK", "Europe")
]

country_df = spark.createDataFrame(country_data, ["country", "continent"])

#### Salting

SALTING: Add a random salt (0 to 2) to the sales_df

In [ ]:
sales_salted = sales_df.withColumn("salt", (rand() * 3).cast("int")) \
    .withColumn("salted_key", concat(col("country"), lit("_"), col("salt")))

In [ ]:
sales_salted.show()

Duplicate country_df for each salt value

In [ ]:
salt_values = spark.createDataFrame([(0,), (1,), (2,)], ["salt"])
country_salted = country_df.crossJoin(salt_values) \
    .withColumn("salted_key", concat(col("country"), lit("_"), col("salt")))

In [ ]:
country_salted.show()

#### Perform the salted join

In [ ]:
sales_alias = sales_salted.alias("sales")
country_alias = country_salted.alias("country")

In [ ]:
joined_df = sales_alias.join(country_alias, on="salted_key", how="inner")
print("🔹 Joined DataFrame:")
joined_df.show()

#### Select final columns clearly

In [ ]:
final_result = joined_df.select(
    col("sales.country").alias("country"),
    col("sales.amount"),
    col("country.continent")
)
print("🔹 Final Result After Removing Salt:")
final_result.show()

### Example 3 (GroupBy/Aggregations)

In [ ]:
from pyspark.sql.functions import lit, count, explode, rand, floor, concat, col, sum

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "auto")

In [ ]:
df_part1 = spark.range(1000).withColumn("partition", lit("partition_1"))
df_part2 = spark.range(100000).withColumn("partition", lit("partition_2"))
df_part3 = spark.range(10000000).withColumn("partition", lit("partition_3"))

df_union = df_part1.union(df_part2).union(df_part3)
df_union.show(2)


In [ ]:
df_grp_by = df_union.groupBy('partition').agg(count("id"))

In [ ]:
df_grp_by.explain('formatted')

In [ ]:
df_grp_by.show()

In [ ]:
df_salted_union = df_union.withColumn('Salt', floor(rand()*10)).withColumn('new_partition', concat(col('partition'), lit('_'), col('Salt')))

In [ ]:
df_salted_union.show(5)

In [ ]:
df_salt_agg = df_salted_union.groupBy('new_partition','partition').agg(count("id")).withColumnRenamed('count(id)', 'id')

In [ ]:
df_salt_agg.drop('new_partition')

In [ ]:
df_salt_agg.withColumn('id', col('id').cast('int')).groupBy('partition').agg(sum("id")).show()

### Example 4 (Distinct)

In [ ]:
df = (
    spark.range(100000000000)
    .withColumn(
        "user_id",
        concat(
            lit("SUPER_HOT_USER_"),
            (col("id") % 1000000000)
        )
    )
)

In [ ]:
df.show(5)

In [ ]:
df.select("user_id").distinct().show()

In [ ]:
salted = df.withColumn(
    "salt",
    floor(rand()*20)
)

partial = (
    salted
    .select("user_id","salt")
    .distinct()
)

final = (
    partial
    .select("user_id")
    .distinct()
)

final.show()

### Example 5 (Window Function)

In case of window function salting is not perfomed directly on window function rather groupby is used wherever possible i.e. when using agg. functions like sum, avg groupby can be used whereas when using rank, row_number is used, salting/groupby is ineffective rather would produce incorrect result.

In [ ]:
from pyspark.sql.functions import lit, expr, sum, floor, rand, concat, substring
from pyspark.sql.window import Window

customers = (
    spark.range(300000000)
         .withColumn(
             "CustomerID",
             expr("""
                 CASE
                     WHEN id < 1000 THEN 'C1'
                     WHEN id < 2000 THEN 'C2'
                     ELSE 'C3'
                 END
             """)
         )
         .withColumn("CustomerName", lit("Customer"))
         .withColumnRenamed("id","Amount")
)

customers.show(5)

In [ ]:
cust_wind = Window.partitionBy('CustomerID')
sum = customers.withColumn('Sum', sum('Amount').over(cust_wind))
sum.show()

In [ ]:
customers_salted = customers.withColumn('salt', floor(rand()*10))
cust_grp = customers_salted.groupBy('salt','CustomerID').agg(sum("Amount").alias('Amount'))
cust_grp = cust_grp.groupBy('CustomerID').agg(sum('Amount').alias('Amount'))
customers = customers.join(cust_grp, on='CustomerID', how='left')
customers.show()

## Broadcast Join

* What it is: If one dataset is significantly smaller than the other (i.e., a "small table" in the join),
you can use a broadcast join to send the small dataset to all worker nodes, instead of shuffling it
with the large dataset.
* How it works: By broadcasting the small dataset, Spark avoids the expensive shuffle operation
that can result in skewed data distribution. This ensures that no single partition ends up with an
overwhelming number of records.
* When to use: Typically when one of the datasets in a join is small enough to fit in memory (e.g.,
a lookup table).

In [ ]:
from pyspark.sql.functions import expr, col, cast

orders = (
    spark.range(1000000)
         .withColumn(
             "Orders",
             expr("""
                 CASE
                     WHEN id < 9000000 THEN '1'
                     WHEN id < 9500000 THEN '2'
                     ELSE '3'
                 END
             """)
         )
)

orders = orders.withColumnRenamed('id', 'Orders_id').withColumnRenamed('Orders','id').withColumn('id', col('id').cast('int'))

In [ ]:
# Change sets to actual dictionaries with keys
customers_dict = [
    {'id': '1', 'name': 'Customer1'},
    {'id': '2', 'name': 'Customer2'},
    {'id': '3', 'name': 'Customer3'}
]

customers = spark.createDataFrame(customers_dict).withColumn('id', col('id').cast('int'))
customers.show()

In [ ]:
from pyspark.sql.functions import broadcast

orders.join(
    broadcast(customers),
    on="id", how='left'
).show()

## Repartitioning

## Skew Join Optimization

* What it is: Spark provides a skew join optimization feature that automatically handles skew in
join operations. When enabled, Spark will attempt to detect and resolve skewed keys during the
shuffle phase.
* How it works: Spark will automatically shuffle and partition the skewed keys in a way that
avoids heavy load on a single executor. This often involves splitting the join into multiple stages
to handle skewed keys separately.

Traditional Spark: <br>
Create Plan --> Execute Plan

AQE: <br>
Create Initial Plan --> Start Execution --> Collect Runtime Statistics --> Modify Plan Dynamically --> Continue Execution

Config: <br>
spark.conf.set(
    "spark.sql.adaptive.enabled",
    "true"
) ----- To enable AQE

spark.conf.set(
    "spark.sql.adaptive.skewJoin.enabled",
    "true"
) ---- To enable skew join handling

In [75]:
from pyspark.sql.functions import expr, col, cast

orders = (
    spark.range(9000000000000000000)
         .withColumn(
             "Orders",
             expr("""
                 CASE
                     WHEN id < 90000000 THEN '1'
                     WHEN id < 95000000 THEN '2'
                     ELSE '3'
                 END
             """)
         )
)

orders = orders.withColumnRenamed('id', 'Orders_id').withColumnRenamed('Orders','id').withColumn('id', col('id').cast('int'))

# Change sets to actual dictionaries with keys
customers_dict = [
    {'id': '1', 'name': 'Customer1'},
    {'id': '2', 'name': 'Customer2'},
    {'id': '3', 'name': 'Customer3'}
]

customers = spark.createDataFrame(customers_dict).withColumn('id', col('id').cast('int'))

In [76]:
spark.conf.get(
    "spark.sql.adaptive.enabled",
    "false"
)

spark.conf.set(
    "spark.sql.adaptive.skewJoin.enabled",
    "false"
)

In [77]:
from datetime import datetime


begin = datetime.now()

orders.join(
    customers,
    on="id", how='left'
).show()

print(datetime.now()-begin)

+---+---------+---------+
| id|Orders_id|     name|
+---+---------+---------+
|  1|        0|Customer1|
|  1|        1|Customer1|
|  1|        2|Customer1|
|  1|        3|Customer1|
|  1|        4|Customer1|
|  1|        5|Customer1|
|  1|        6|Customer1|
|  1|        7|Customer1|
|  1|        8|Customer1|
|  1|        9|Customer1|
|  1|       10|Customer1|
|  1|       11|Customer1|
|  1|       12|Customer1|
|  1|       13|Customer1|
|  1|       14|Customer1|
|  1|       15|Customer1|
|  1|       16|Customer1|
|  1|       17|Customer1|
|  1|       18|Customer1|
|  1|       19|Customer1|
+---+---------+---------+
only showing top 20 rows
0:00:01.282084


In [78]:
spark.conf.get(
    "spark.sql.adaptive.enabled",
    "true"
)

spark.conf.set(
    "spark.sql.adaptive.skewJoin.enabled",
    "true"
)

In [79]:
from datetime import datetime


begin = datetime.now()

orders.join(
    customers,
    on="id", how='left'
).show()

print(datetime.now()-begin)

+---+---------+---------+
| id|Orders_id|     name|
+---+---------+---------+
|  1|        0|Customer1|
|  1|        1|Customer1|
|  1|        2|Customer1|
|  1|        3|Customer1|
|  1|        4|Customer1|
|  1|        5|Customer1|
|  1|        6|Customer1|
|  1|        7|Customer1|
|  1|        8|Customer1|
|  1|        9|Customer1|
|  1|       10|Customer1|
|  1|       11|Customer1|
|  1|       12|Customer1|
|  1|       13|Customer1|
|  1|       14|Customer1|
|  1|       15|Customer1|
|  1|       16|Customer1|
|  1|       17|Customer1|
|  1|       18|Customer1|
|  1|       19|Customer1|
+---+---------+---------+
only showing top 20 rows
0:00:00.626378


## Custom Partitioning

## Avoid Shuffles

## Data Sampling